In [1]:
import warnings
warnings.simplefilter(action='ignore')
import os, joblib
import numpy as np
import pandas as pd
import polars as pl
import kaggle_evaluation.default_inference_server
from catboost import CatBoostRegressor, Pool
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import train_test_split

In [2]:
train = pd.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv')

def preprocessing(data, typ):
    main_features = ['E1','E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 
                     'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',  
                     'S1', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S12', 
                     "I2", "P8", "P9", "P10", "P12", "P13",
                     
                     "M*", "E*", "I*", "P*", "V*", "S*", "D*",
                     
                     "M*_P*_V*", 'M*_E*_S*', 'M*_P*_I*_V*',
                     'V*_M*_E*_I*', 'V*_S*_D*',
                     'E*_I*_V*_D*', 'M*_V*_S*_D*',
                     'P*_V*_S*', 'P*_M*_D*',
                     'M*_E*_P*_S*', 'M*_E*_I*_P*_V*_S*_D*'
                    ]
    
    data['M*'] = data[[f'M{i}' for i in range(1, 19)]].sum(axis=1).to_numpy()
    data['E*'] = data[[f'E{i}' for i in range(1, 21)]].sum(axis=1).to_numpy()
    data['I*'] = data[[f'I{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()
    data['P*'] = data[['P8', 'P9', 'P10', 'P12', 'P13']].sum(axis=1).to_numpy()
    data['V*'] = data[[f'V{i}' for i in range(1, 14)]].sum(axis=1).to_numpy()
    data['S*'] = data[[f'S{i}' for i in range(1, 13)]].sum(axis=1).to_numpy()
    data['D*'] = data[[f'D{i}' for i in range(1, 10)]].sum(axis=1).to_numpy()

    data[f'ME_prod_M*_E*'] = data['M*'] * data['E*']
    data[f'ME_ratio_M*_E*'] = data['M*'] / (data['E*'] + 1e-8)
            
    main_features.append(f'ME_prod_M*_E*')
    main_features.append(f'ME_ratio_M*_E*')
            
    data[f'MI_prod_M*_I*'] = data['M*'] * data['I*']
    data[f'MI_ratio_M*_I*'] = data['M*'] / (data['I*'] + 1e-8)
    data[f'MI_spread_M*_I*'] = data['M*'] - data['I*']
    
    main_features.append(f'MI_prod_M*_I*')
    main_features.append(f'MI_ratio_M*_I*')
    main_features.append(f'MI_spread_M*_I*')
        
    data[f'MP_prod_M*_P*'] = data['M*'] * data['P*']
    data[f'MP_ratio_M*_P*'] = data['M*'] / (data['P*'] + 1e-8)
    
    main_features.append(f'MP_prod_M*_P*')
    main_features.append(f'MP_ratio_M*_P*')
    
    data[f'MV_ratio_M*_V*'] = data['M*'] / (data['V*'] + 1e-8)
    data[f'MV_prod_M*_V*'] = data['M*'] * data['V*']
                
    main_features.append(f'MV_ratio_M*_V*')
    main_features.append(f'MV_prod_M*_V*')
        
    data[f'MS_prod_M*_S*'] = data['M*'] * data['S*']
    data[f'MS_weighted_M*_S*'] = data['M*'] * (1 + data['S*'])
    main_features.append(f'MS_prod_M*_S*')
    main_features.append(f'MS_weighted_M*_S*')
        
    data[f'EI_diff_E*_I*'] = data['E*'] - data['I*']
    data[f'EI_ratio_E*_I*'] = data['E*'] / (data['I*'] + 1e-8)
    data[f'EI_prod_E*_I*'] = data['E*'] * data['I*']
    main_features.append(f'EI_diff_E*_I*')
    main_features.append(f'EI_ratio_E*_I*')
    main_features.append(f'EI_prod_E*_I*')
        
    data[f'EP_prod_E*_P*'] = data['E*'] * data['P*']
    data[f'EP_ratio_E*_P*'] = data['E*'] / (data['P*'] + 1e-8)
    main_features.append(f'EP_prod_E*_P*')
    main_features.append(f'EP_ratio_E*_P*')
        
    data[f'EV_prod_E*_V*'] = data['E*'] * data['V*']
    data[f'EV_uncertainty_E*_V*'] = abs(data['E*']) * data['V*']
    main_features.append(f'EV_prod_E*_V*')
    main_features.append(f'EV_uncertainty_E*_V*')
        
    data[f'ES_prod_E*_S*'] = data['E*'] * data['S*']
    data[f'ES_diff_E*_S*'] = data['E*'] - data['S*']
    main_features.append(f'ES_prod_E*_S*')
    main_features.append(f'ES_diff_E*_S*')
        
    data[f'IP_prod_I*_P*'] = data['I*'] * data['P*']
    data[f'IP_discount_I*_P*'] = data['P*'] / (1 + data['I*'] + 1e-8)
    main_features.append(f'IP_prod_I*_P*')
    main_features.append(f'IP_discount_I*_P*')
    
    data[f'IV_prod_I*_V*'] = data['I*'] * data['V*']
    main_features.append(f'IV_prod_I*_V*')
        
    data[f'IS_prod_I*_S*'] = data['I*'] * data['S*']
    main_features.append(f'IS_prod_I*_S*')
        
    data[f'PV_prod_P*_V*'] = data['P*'] * data['V*']
    data[f'PV_stability_P*_V*'] = data['P*'] / (data['V*'] + 1e-8)
    main_features.append(f'PV_prod_P*_V*')
    main_features.append(f'PV_stability_P*_V*')
        
    data[f'PS_prod_P*_S*'] = data['P*'] * data['S*']
    data[f'PS_contrarian_P*_S*'] = -data['P*'] * data['S*']
    main_features.append(f'PS_prod_P*_S*')
    main_features.append(f'PS_contrarian_P*_S*')
        
    data[f'VS_prod_V*_S*'] = data['V*'] * data['S*']
    data[f'VS_panic_V*_S*'] = data['V*'] * abs(data['S*'])
    main_features.append(f'VS_prod_V*_S*')
    main_features.append(f'VS_panic_V*_S*')

    data['M*_P*_V*'] = data['M*'] + data['P*'] + data['V*']
    data['M*_E*_S*'] = data['M*'] + data['E*'] + data['S*'] 
    data['M*_P*_I*_V*'] = data['M*'] + data['P*'] + data['I*'] + data['V*'] 

    data['V*_M*_E*_I*'] = data['V*'] + data['M*'] + data['E*'] + data['I*'] 
    data['V*_S*_D*'] = data['V*'] + data['S*'] + data['D*'] 

    data['E*_I*_V*_D*'] = data['E*'] + data['I*'] + data['V*'] + data['D*']
    data['M*_V*_S*_D*'] = data['M*'] + data['V*'] + data['S*'] + data['D*'] 

    data['P*_V*_S*'] = data['P*'] + data['V*'] + data['S*'] 
    data['P*_M*_D*'] = data['P*'] + data['M*'] + data['D*']

    data['M*_E*_P*_S*'] = data['M*'] + data['E*'] + data['P*'] + data['S*']
    data['M*_E*_I*_P*_V*_S*_D*'] = data['M*'] + data['E*'] + data['I*'] + data['P*'] + data['V*'] + data['S*'] + data['D*']
    
    if typ == "train":
        data = data[main_features + ["forward_returns"]].copy()
        data = data.rename(columns={'forward_returns': 'target'})
    else: 
        data = data[main_features].copy()
    
    if 'target' in data.columns:
        data = data.dropna()

    for col in data.columns:
        if col != 'target':
            data[col].fillna(data[col].mean(), inplace=True)
    
    return data

train_processed = preprocessing(train, "train")

train_x = train_processed.drop(columns=["target"])
train_y = train_processed['target']

In [3]:
train_processed

,E1,E10,E11,E12,E13,E14,E15,E16,E17,E18,...,IP_discount_I*_P*,IV_prod_I*_V*,IS_prod_I*_S*,PV_prod_P*_V*,PV_stability_P*_V*,PS_prod_P*_S*,PS_contrarian_P*_S*,VS_prod_V*_S*,VS_panic_V*_S*,target
6969,0.682135,0.017196,0.007937,0.007937,0.007937,0.007937,0.048280,1.148054,1.303514,1.046752,...,0.415121,39.293238,13.819007,18.370000,0.747024,6.460530,-6.460530,8.648361,8.648361,0.001145
6970,0.681101,0.016865,0.007606,0.007606,0.007606,0.007606,0.008267,1.146588,1.303443,1.047816,...,0.720893,36.652103,17.124485,29.821070,1.341544,13.932911,-13.932911,10.385726,10.385726,0.004738
6971,0.680068,0.016534,0.007275,0.007275,0.007275,0.007275,0.007937,1.145124,1.303371,1.048881,...,0.813072,38.321956,22.500644,35.111530,1.485426,20.615650,-20.615650,13.878609,13.878609,0.006016
6972,0.679035,0.016204,0.006944,0.006944,0.006944,0.006944,0.007606,1.120467,2.311946,1.049948,...,0.707903,33.359403,25.236451,26.727384,1.382853,20.219316,-20.219316,14.621450,14.621450,0.001414
6973,0.678003,0.015873,0.006614,0.006614,0.006614,0.006614,0.007275,1.119052,2.308384,1.051017,...,0.671389,32.548385,21.987350,24.789544,1.295489,16.746035,-16.746035,12.926420,12.926420,-0.007182
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8985,1.565379,0.184524,0.019180,0.019180,0.005952,0.005952,0.911376,-0.083496,-0.572447,0.223638,...,0.975153,14.998988,12.413334,18.013558,1.492962,14.908227,-14.908227,9.985670,9.985670,0.002457
8986,1.562946,0.184193,0.018849,0.018849,0.005622,0.005622,0.911706,-0.083542,-0.572080,0.222910,...,1.068629,13.720627,21.347952,18.137762,1.714749,28.220582,-28.220582,16.457558,16.457558,0.002312
8987,1.560520,0.183862,0.018519,0.018519,0.005291,0.005291,0.912037,-0.083874,-0.572016,0.222211,...,0.779983,11.527504,17.911727,11.147219,1.459002,17.320831,-17.320831,11.871701,11.871701,0.002891
8988,1.558102,0.183532,0.018188,0.018188,0.004960,0.004960,0.912368,-0.084206,-0.571952,0.221513,...,1.205378,12.385137,7.635535,18.450361,2.161599,11.374792,-11.374792,5.262211,5.262211,0.008310


In [4]:
import numpy as np
import joblib
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, mean_squared_error

class ResidualBoostingEnsemble:
    
    def __init__(self, base_params, n_models=3):
        self.base_params = base_params
        self.n_models = n_models
        self.models = []
        
    def fit(self, train_x, train_y, test_x=None, test_y=None):
        current_train_pred = np.zeros(len(train_y))
        
        for i in range(self.n_models):
            print(f'Training Model {i+1}...')
            
            if i == 0:
                target = train_y
            else:
                residuals = train_y - current_train_pred
                target = residuals
            
            model = LGBMRegressor(**self.base_params)
            model.fit(train_x, target)
            self.models.append(model)
            
            pred = model.predict(train_x)
            current_train_pred += pred
            
            train_mape = mean_absolute_error(train_y, current_train_pred)
            print(f'Cumulative Training MAPE: {train_mape:.4f}')
        
        return self
    
    def predict(self, X):
        predictions = np.zeros(len(X))
        for model in self.models:
            predictions += model.predict(X)
        return predictions
    
    def save(self, filepath):
        joblib.dump(self, filepath)
        print(f'Model saved to {filepath}')
    
    @staticmethod
    def load(filepath):
        return joblib.load(filepath)
    
    def evaluate(self, X, y):
        predictions = self.predict(X)
        mape = mean_absolute_percentage_error(y, predictions)
        mae = mean_absolute_error(y, predictions)
        rmse = np.sqrt(mean_squared_error(y, predictions))
        
        print(f'MAPE: {mape:.4f}')
        print(f'MAE: {mae:.4f}')
        print(f'RMSE: {rmse:.4f}')
        
        return {'mape': mape, 'mae': mae, 'rmse': rmse}


LGBM_R_parm = {
               'boosting_type': 'gbdt', 
               'colsample_bytree': 0.95, 
               'learning_rate': 0.08,
               'max_bin': 100,
               'max_depth': 12,
               'metric': 'regression_l2', 
               'min_child_samples': 60,
               'min_data_in_leaf': 20, 
               'n_estimators': 7000,
               'num_leaves': 50,
               'objective': 'mean_absolute_error', 
               'reg_alpha': 0.8,
               'reg_lambda': 3.5, 
               'subsample': 0.75, 
               'verbosity': -1
              }

ensemble = ResidualBoostingEnsemble(base_params=LGBM_R_parm, n_models=3)
ensemble.fit(train_x, train_y)

ensemble.save('LGBM_Residual_Ensemble.pkl')

print('\nFinal Training Evaluation:')
ensemble.evaluate(train_x, train_y)

LGBM_R_model = joblib.load('LGBM_Residual_Ensemble.pkl')

Training Model 1...
Cumulative Training MAPE: 0.0018
Training Model 2...
Cumulative Training MAPE: 0.0017
Training Model 3...
Cumulative Training MAPE: 0.0016
Model saved to LGBM_Residual_Ensemble.pkl

Final Training Evaluation:
MAPE: 92397018.6396
MAE: 0.0016
RMSE: 0.0042


In [5]:
import numpy as np
import pandas as pd
from typing import Optional, Tuple, Dict, List
from collections import deque
from scipy.optimize import minimize, Bounds

SIGNAL_MULTIPLIER = 50000.0
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0

# Calibrated constants
TARGET_VOL_RATIO = 1.2
MARKET_VOL_BASELINE = 0.16
DAILY_VOL_BASELINE = MARKET_VOL_BASELINE / np.sqrt(252)


class VolatilityForecaster:
    """Forecast volatility using EWMA/GARCH-style models"""
    
    def __init__(self, lambda_decay=0.94):
        self.lambda_decay = lambda_decay
        self.vol_ewma = None
    
    def forecast(self, returns_history: List[float], horizon: int = 1) -> float:
        """
        Forecast volatility using exponentially weighted moving average
        """
        if len(returns_history) < 5:
            return DAILY_VOL_BASELINE
        
        returns = np.array(returns_history[-100:])  # Use last 100 days
        
        # EWMA weights
        n = len(returns)
        weights = np.array([
            (1 - self.lambda_decay) * (self.lambda_decay ** i) 
            for i in range(n)
        ])
        weights = weights[::-1] / weights.sum()
        
        # Weighted variance
        variance = np.sum(weights * (returns ** 2))
        vol = np.sqrt(variance)
        
        # Update EWMA
        if self.vol_ewma is None:
            self.vol_ewma = vol
        else:
            self.vol_ewma = self.lambda_decay * self.vol_ewma + (1 - self.lambda_decay) * vol
        
        # Scale to horizon
        vol_forecast = self.vol_ewma * np.sqrt(horizon)
        
        return vol_forecast


class ModelUncertaintyEstimator:
    """Estimate prediction uncertainty using various methods"""
    
    def __init__(self):
        self.train_feature_stats = {}
    
    def set_training_stats(self, train_df: pd.DataFrame):
        """Store training data statistics for OOD detection"""
        for col in train_df.columns:
            if train_df[col].dtype in [np.float64, np.float32, np.int64, np.int32]:
                self.train_feature_stats[col] = {
                    'mean': train_df[col].mean(),
                    'std': train_df[col].std(),
                    'min': train_df[col].min(),
                    'max': train_df[col].max()
                }
    
    def get_feature_based_uncertainty(self, test_features: pd.Series) -> float:
        """Calculate uncertainty based on feature out-of-distribution"""
        if not self.train_feature_stats:
            return 0.3  # Default moderate uncertainty
        
        z_scores = []
        for col, value in test_features.items():
            if col in self.train_feature_stats:
                stats = self.train_feature_stats[col]
                z_score = abs((value - stats['mean']) / (stats['std'] + 1e-8))
                z_scores.append(min(z_score, 5.0))  # Cap extreme values
        
        if not z_scores:
            return 0.3
        
        # Average z-score as OOD metric
        avg_z_score = np.mean(z_scores)
        uncertainty = np.tanh(avg_z_score / 3)  # Scale to [0,1]
        
        return uncertainty
    
    def get_prediction_interval_uncertainty(
        self, 
        model, 
        X: pd.DataFrame,
        n_bootstrap: int = 10
    ) -> float:
        """
        Estimate uncertainty using bootstrap predictions if available
        """
        if not hasattr(model, 'predict'):
            return 0.3
        
        try:
            # Simple variance estimation from single prediction
            # In practice, you'd want multiple models or quantile regression
            base_pred = model.predict(X)[0]
            
            # Rough uncertainty estimate based on prediction magnitude
            uncertainty = min(abs(base_pred) * 10, 0.5)
            
            return uncertainty
        except:
            return 0.3


class MarketRegimeDetector:
    """Detect market regimes from features"""
    
    @staticmethod
    def detect_regime(features: pd.Series) -> Dict[str, str]:
        """
        Detect market regime: trend, volatility, risk sentiment
        """
        regime = {
            'trend': 'neutral',
            'volatility': 'normal',
            'risk_sentiment': 'neutral'
        }
        
        # Trend detection - look for momentum/moving average features
        momentum_cols = [col for col in features.index if 'momentum' in col.lower() or 'ma_' in col.lower()]
        if momentum_cols:
            avg_momentum = features[momentum_cols].mean()
            if avg_momentum > 0.02:
                regime['trend'] = 'bull'
            elif avg_momentum < -0.02:
                regime['trend'] = 'bear'
        
        # Volatility regime - look for volatility features
        vol_cols = [col for col in features.index if 'vol' in col.lower() or 'std' in col.lower()]
        if vol_cols:
            avg_vol = features[vol_cols].mean()
            if avg_vol > 0.025:
                regime['volatility'] = 'high'
            elif avg_vol < 0.012:
                regime['volatility'] = 'low'
        
        # Risk sentiment - look for VIX or similar
        risk_cols = [col for col in features.index if 'vix' in col.lower() or 'risk' in col.lower()]
        if risk_cols:
            avg_risk = features[risk_cols].mean()
            if avg_risk > 0.5:  # Normalized
                regime['risk_sentiment'] = 'risk_off'
            elif avg_risk < -0.5:
                regime['risk_sentiment'] = 'risk_on'
        
        return regime
    
    @staticmethod
    def adjust_signal_for_regime(signal: float, regime: Dict[str, str]) -> float:
        """Adjust signal based on detected market regime"""
        adjustment = 1.0
        
        # High volatility: reduce exposure
        if regime['volatility'] == 'high':
            adjustment *= 0.7
        elif regime['volatility'] == 'low':
            adjustment *= 1.1
        
        # Risk-off: be conservative with longs
        if regime['risk_sentiment'] == 'risk_off' and signal > 1.0:
            adjustment *= 0.75
        
        # Strong trends: allow more conviction
        if regime['trend'] in ['bull', 'bear']:
            adjustment *= 1.15
        
        # Apply adjustment
        adjusted = 1.0 + (signal - 1.0) * adjustment
        return np.clip(adjusted, MIN_SIGNAL, MAX_SIGNAL)


class DrawdownController:
    """Reduce exposure after significant drawdowns"""
    
    def __init__(self, max_dd_threshold: float = 0.10):
        self.cumulative_return = 1.0
        self.peak_value = 1.0
        self.current_dd = 0.0
        self.max_dd_threshold = max_dd_threshold
    
    def update(self, period_return: float):
        """Update with new period return"""
        self.cumulative_return *= (1 + period_return)
        self.peak_value = max(self.peak_value, self.cumulative_return)
        
        if self.peak_value > 0:
            self.current_dd = (self.peak_value - self.cumulative_return) / self.peak_value
    
    def get_exposure_scalar(self) -> float:
        """Get exposure reduction factor based on current drawdown"""
        if self.current_dd > self.max_dd_threshold:
            # Reduce exposure proportionally to excess drawdown
            excess_dd = self.current_dd - self.max_dd_threshold
            scalar = max(0.3, 1 - excess_dd * 5)
            return scalar
        return 1.0
    
    def reset(self):
        """Reset drawdown tracking"""
        self.cumulative_return = 1.0
        self.peak_value = 1.0
        self.current_dd = 0.0


class BayesianSignalUpdater:
    """Update signals using Bayesian inference"""
    
    def __init__(self, prior_mean: float = 0.0, prior_std: float = 0.01):
        self.prior_mean = prior_mean
        self.prior_std = prior_std
    
    def update_signal(
        self, 
        model_prediction: float, 
        prediction_std: float
    ) -> Tuple[float, float]:
        """
        Bayesian update: combine prior belief with model prediction
        Returns: (posterior_mean, posterior_std)
        """
        # Prior and likelihood precisions
        prior_precision = 1 / (self.prior_std ** 2)
        likelihood_precision = 1 / (prediction_std ** 2)
        
        # Posterior precision and variance
        posterior_precision = prior_precision + likelihood_precision
        posterior_std = 1 / np.sqrt(posterior_precision)
        
        # Posterior mean (weighted average)
        posterior_mean = (
            (prior_precision * self.prior_mean + 
             likelihood_precision * model_prediction) / posterior_precision
        )
        
        # Update prior for next iteration (with decay)
        self.prior_mean = posterior_mean * 0.9
        self.prior_std = min(posterior_std * 1.1, 0.05)
        
        return posterior_mean, posterior_std


class SignalSmoother:
    """Apply exponential smoothing to reduce signal noise"""
    
    def __init__(self, alpha: float = 0.3):
        self.alpha = alpha
        self.smoothed_history = deque(maxlen=10)
    
    def smooth(self, current_signal: float) -> float:
        """Apply exponential smoothing"""
        if len(self.smoothed_history) == 0:
            smoothed = current_signal
        else:
            last_smoothed = self.smoothed_history[-1]
            smoothed = self.alpha * current_signal + (1 - self.alpha) * last_smoothed
        
        self.smoothed_history.append(smoothed)
        return smoothed


class TransactionCostController:
    """Control trading frequency to minimize transaction costs"""
    
    def __init__(self, cost_per_trade: float = 0.001, min_change_threshold: float = 0.1):
        self.cost_per_trade = cost_per_trade
        self.min_change_threshold = min_change_threshold
        self.last_signal = 1.0
    
    def apply_cost_penalty(self, current_signal: float) -> float:
        """
        Only change position if signal change exceeds threshold
        """
        signal_change = abs(current_signal - self.last_signal)
        
        # Require minimum change to trade
        if signal_change < self.min_change_threshold:
            return self.last_signal
        
        # Apply cost adjustment
        direction = np.sign(current_signal - self.last_signal)
        cost_adjusted = current_signal - direction * self.cost_per_trade
        cost_adjusted = np.clip(cost_adjusted, MIN_SIGNAL, MAX_SIGNAL)
        
        self.last_signal = cost_adjusted
        return cost_adjusted
    
    def reset(self, signal: float = 1.0):
        """Reset to initial state"""
        self.last_signal = signal


class AdvancedSignalCalibrator:
    """
    Advanced signal calibration with multiple enhancement techniques
    """
    
    def __init__(
        self,
        kelly_fraction: float = 0.25,
        vol_target_ratio: float = 1.2,
        use_regime_adjustment: bool = True,
        use_drawdown_control: bool = True,
        use_bayesian_update: bool = False,
        use_signal_smoothing: bool = False,
        use_transaction_cost: bool = False
    ):
        self.kelly_fraction = kelly_fraction
        self.vol_target_ratio = vol_target_ratio
        self.use_regime_adjustment = use_regime_adjustment
        self.use_drawdown_control = use_drawdown_control
        self.use_bayesian_update = use_bayesian_update
        self.use_signal_smoothing = use_signal_smoothing
        self.use_transaction_cost = use_transaction_cost
        
        # Initialize components
        self.uncertainty_estimator = ModelUncertaintyEstimator()
        self.vol_forecaster = VolatilityForecaster()
        self.drawdown_controller = DrawdownController() if use_drawdown_control else None
        self.bayesian_updater = BayesianSignalUpdater() if use_bayesian_update else None
        self.signal_smoother = SignalSmoother() if use_signal_smoothing else None
        self.transaction_controller = TransactionCostController() if use_transaction_cost else None
        
        # History tracking
        self.return_history = deque(maxlen=100)
        self.prediction_history = deque(maxlen=100)
    
    def estimate_prediction_uncertainty(
        self, 
        raw_prediction: float,
        features: Optional[pd.Series] = None
    ) -> float:
        """
        Enhanced uncertainty estimation combining multiple sources
        """
        # Base uncertainty from prediction magnitude
        abs_pred = abs(raw_prediction)
        
        if abs_pred < 0.001:
            base_uncertainty = 0.8
        elif abs_pred < 0.01:
            base_uncertainty = 0.3
        elif abs_pred < 0.05:
            base_uncertainty = 0.15
        else:
            base_uncertainty = 0.15 + min(0.6, (abs_pred - 0.05) * 10)
        
        # Add feature-based uncertainty if available
        if features is not None:
            feature_uncertainty = self.uncertainty_estimator.get_feature_based_uncertainty(features)
            # Combine uncertainties
            combined_uncertainty = (base_uncertainty + feature_uncertainty) / 2
            return min(combined_uncertainty, 0.9)
        
        return base_uncertainty
    
    def calculate_base_exposure(
        self, 
        raw_prediction: float,
        uncertainty: float
    ) -> float:
        """Convert raw prediction to base exposure"""
        confidence = 1.0 - uncertainty
        base_signal = 1.0 + np.tanh(raw_prediction * 100) * confidence
        return base_signal
    
    def apply_volatility_targeting(
        self, 
        base_signal: float,
        forecasted_vol: Optional[float] = None
    ) -> float:
        """
        Scale signal to target volatility ratio
        """
        exposure = abs(base_signal - 1.0)
        
        # Adjust for forecasted volatility
        if forecasted_vol is not None and forecasted_vol > 0:
            vol_adjustment = DAILY_VOL_BASELINE / forecasted_vol
            vol_adjustment = np.clip(vol_adjustment, 0.5, 2.0)
        else:
            vol_adjustment = 1.0
        
        # Target ratio scaling
        target_safe_ratio = self.vol_target_ratio * 0.95  # Safety margin
        max_safe_exposure = target_safe_ratio - 1.0
        
        if exposure > max_safe_exposure:
            scaling_factor = max_safe_exposure / exposure
            scaled_signal = 1.0 + (base_signal - 1.0) * scaling_factor * vol_adjustment
        else:
            scaled_signal = 1.0 + (base_signal - 1.0) * vol_adjustment
        
        return scaled_signal
    
    def apply_kelly_criterion(
        self, 
        signal: float,
        raw_prediction: float,
        prediction_std: Optional[float] = None
    ) -> float:
        """Apply Kelly criterion for optimal position sizing"""
        if prediction_std is None:
            prediction_std = DAILY_VOL_BASELINE * 1.5
        
        if abs(raw_prediction) < 1e-6:
            return signal
        
        # Kelly fraction = edge / variance
        variance = prediction_std ** 2
        kelly_fraction = raw_prediction / variance if variance > 0 else 0
        kelly_adjustment = 1.0 + (kelly_fraction * self.kelly_fraction)
        
        # Blend with original (70% kelly, 30% original)
        kelly_weight = 0.7
        adjusted = kelly_weight * kelly_adjustment + (1 - kelly_weight) * signal
        
        return adjusted
    
    def apply_asymmetric_risk(
        self, 
        signal: float,
        downside_scale: float = 1.5
    ) -> float:
        """
        Apply asymmetric risk adjustment - more conservative on shorts
        """
        if signal < 1.0:
            exposure = signal - 1.0
            adjusted_exposure = exposure / downside_scale
            signal = 1.0 + adjusted_exposure
        
        return signal
    
    def process_signal(
        self,
        raw_prediction: float,
        features: Optional[pd.Series] = None,
        actual_return: Optional[float] = None
    ) -> float:
        """
        Complete signal processing pipeline with all enhancements
        """
        # Update history
        self.prediction_history.append(raw_prediction)
        if actual_return is not None:
            self.return_history.append(actual_return)
            if self.drawdown_controller:
                self.drawdown_controller.update(actual_return)
        
        # Step 1: Estimate uncertainty
        uncertainty = self.estimate_prediction_uncertainty(raw_prediction, features)
        
        # Step 2: Bayesian update (if enabled)
        if self.use_bayesian_update and self.bayesian_updater:
            prediction_std = uncertainty * abs(raw_prediction)
            raw_prediction, prediction_std = self.bayesian_updater.update_signal(
                raw_prediction, prediction_std
            )
        
        # Step 3: Base exposure calculation
        base_signal = self.calculate_base_exposure(raw_prediction, uncertainty)
        
        # Step 4: Volatility forecasting and targeting
        forecasted_vol = None
        if len(self.return_history) > 10:
            forecasted_vol = self.vol_forecaster.forecast(list(self.return_history))
        
        signal = self.apply_volatility_targeting(base_signal, forecasted_vol)
        
        # Step 5: Kelly criterion
        signal = self.apply_kelly_criterion(signal, raw_prediction)
        
        # Step 6: Asymmetric risk adjustment
        signal = self.apply_asymmetric_risk(signal)
        
        # Step 7: Regime adjustment
        if self.use_regime_adjustment and features is not None:
            regime = MarketRegimeDetector.detect_regime(features)
            signal = MarketRegimeDetector.adjust_signal_for_regime(signal, regime)
        
        # Step 8: Drawdown control
        if self.use_drawdown_control and self.drawdown_controller:
            exposure_scalar = self.drawdown_controller.get_exposure_scalar()
            signal = 1.0 + (signal - 1.0) * exposure_scalar
        
        # Step 9: Signal smoothing
        if self.use_signal_smoothing and self.signal_smoother:
            signal = self.signal_smoother.smooth(signal)
        
        # Step 10: Transaction cost control
        if self.use_transaction_cost and self.transaction_controller:
            signal = self.transaction_controller.apply_cost_penalty(signal)
        
        # Final clip
        signal = np.clip(signal, MIN_SIGNAL, MAX_SIGNAL)
        
        return signal


# Global calibrator instance (configure as needed)
global_calibrator = AdvancedSignalCalibrator(
    kelly_fraction=0.25,
    vol_target_ratio=1.2,
    use_regime_adjustment=True,
    use_drawdown_control=False,  # Disabled in inference (no actual returns)
    use_bayesian_update=False,   # Can enable if you have prediction std
    use_signal_smoothing=False,  # Disabled in inference (state issues)
    use_transaction_cost=False   # Disabled in inference (state issues)
)


def predict(test: pl.DataFrame) -> float:
    """
    Enhanced prediction function with advanced signal processing
    """
    try:
        test_df = test.to_pandas()
        
        # Preprocessing
        test_processed = preprocessing(test_df, 'inference')
        
        # Get model prediction
        LGBM_R_y_pred = np.float64(LGBM_R_model.predict(test_processed)[0])
        
        # Extract features for regime detection (if available)
        try:
            features = test_processed.iloc[0]
        except:
            features = None
        
        # Process signal through calibrator
        signal = global_calibrator.process_signal(
            LGBM_R_y_pred,
            features=features,
            actual_return=None
        )
        
        print(f"Raw pred: {LGBM_R_y_pred:.6f} -> Signal: {signal:.6f}")
        return np.float64(signal)
        
    except Exception as e:
        print(f"Prediction error: {e}")
        import traceback
        traceback.print_exc()
        return 1.0


# Offline optimization and analysis tools
def optimize_signal_parameters(
    validation_predictions: np.ndarray,
    validation_returns: np.ndarray,
    validation_risk_free: np.ndarray,
    validation_features: Optional[pd.DataFrame] = None
) -> Dict[str, float]:
    """
    Optimize signal processing parameters on validation set
    """
    param_grid = {
        'kelly_fraction': [0.1, 0.25, 0.5],
        'vol_target_ratio': [1.1, 1.2, 1.3],
        'use_regime': [True, False]
    }
    
    best_sharpe = -np.inf
    best_params = None
    
    for kelly in param_grid['kelly_fraction']:
        for vol_target in param_grid['vol_target_ratio']:
            for use_regime in param_grid['use_regime']:
                
                calibrator = AdvancedSignalCalibrator(
                    kelly_fraction=kelly,
                    vol_target_ratio=vol_target,
                    use_regime_adjustment=use_regime,
                    use_drawdown_control=False,
                    use_bayesian_update=False,
                    use_signal_smoothing=False,
                    use_transaction_cost=False
                )
                
                # Generate signals
                signals = []
                for i, pred in enumerate(validation_predictions):
                    feats = validation_features.iloc[i] if validation_features is not None else None
                    signal = calibrator.process_signal(pred, features=feats)
                    signals.append(signal)
                
                # Calculate performance
                metrics = backtest_signal_strategy(
                    np.array(signals),
                    validation_returns,
                    validation_risk_free
                )
                
                if metrics['adjusted_sharpe'] > best_sharpe:
                    best_sharpe = metrics['adjusted_sharpe']
                    best_params = {
                        'kelly_fraction': kelly,
                        'vol_target_ratio': vol_target,
                        'use_regime': use_regime,
                        'sharpe': best_sharpe
                    }
    
    return best_params


def backtest_signal_strategy(
    signals: np.ndarray,
    forward_returns: np.ndarray,
    risk_free_rate: np.ndarray
) -> Dict[str, float]:
    """
    Backtest strategy and compute competition metrics
    """
    # Calculate strategy returns
    strategy_returns = (
        risk_free_rate * (1 - signals) +
        forward_returns * signals
    )
    
    # Performance metrics
    excess_returns = strategy_returns - risk_free_rate
    cumulative_return = (1 + excess_returns).prod()
    mean_excess_return = cumulative_return ** (1 / len(excess_returns)) - 1
    
    strategy_std = strategy_returns.std()
    sharpe = mean_excess_return / strategy_std * np.sqrt(252) if strategy_std > 0 else 0
    
    # Volatility metrics
    strategy_vol = strategy_std * np.sqrt(252) * 100
    market_std = forward_returns.std()
    market_vol = market_std * np.sqrt(252) * 100
    vol_ratio = strategy_vol / market_vol if market_vol > 0 else 0
    
    # Competition penalties
    excess_vol = max(0, vol_ratio - 1.2)
    vol_penalty = 1 + excess_vol
    
    market_excess_returns = forward_returns - risk_free_rate
    market_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess = market_cumulative ** (1 / len(market_excess_returns)) - 1
    
    return_gap = max(0, (market_mean_excess - mean_excess_return) * 100 * 252)
    return_penalty = 1 + (return_gap ** 2) / 100
    
    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    
    return {
        'sharpe': float(sharpe),
        'adjusted_sharpe': float(adjusted_sharpe),
        'strategy_vol': float(strategy_vol),
        'market_vol': float(market_vol),
        'vol_ratio': float(vol_ratio),
        'vol_penalty': float(vol_penalty),
        'return_penalty': float(return_penalty),
        'mean_signal': float(signals.mean()),
        'signal_std': float(signals.std()),
        'max_exposure': float(np.abs(signals - 1.0).max()),
        'total_return': float((1 + strategy_returns).prod() - 1)
    }


# Inference server setup
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        ('/kaggle/input/hull-tactical-market-prediction/',)
    )

Raw pred: -0.005964 -> Signal: 0.000000
Raw pred: -0.005427 -> Signal: 0.000000
Raw pred: 0.005419 -> Signal: 2.000000
Raw pred: 0.007653 -> Signal: 2.000000
Raw pred: -0.002896 -> Signal: 0.000000
Raw pred: 0.002457 -> Signal: 2.000000
Raw pred: 0.002311 -> Signal: 2.000000
Raw pred: 0.002891 -> Signal: 2.000000
Raw pred: 0.008308 -> Signal: 2.000000
Raw pred: 0.000099 -> Signal: 1.077009
